# Small Language Model from Scratch
### Modified for Intel Arc XPU / NVIDIA CUDA / CPU

> **Changes from original:**
> - Universal device detection (XPU → CUDA → CPU fallback)
> - `torch.load` uses `map_location=device` (no Colab-specific code)
> - `GradScaler` uses `torch.amp` (not the deprecated `torch.cuda.amp`)
> - `autocast` uses the detected `device_type` string
> - Colab `runtime.unassign()` cell removed
> - `batch_size` starts at 8; increase to 16 → 32 as VRAM allows


## Cell 0 — Install dependencies

In [ ]:
# Run once. On Colab prefix each line with !
# pip install datasets tiktoken tqdm matplotlib


## Cell 1 — Device detection (XPU / CUDA / CPU)

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = torch.cuda.get_device_name(0)

elif hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")
    device_name = torch.xpu.get_device_name(0)

else:
    device = torch.device("cpu")
    device_name = "CPU"

device_str = str(device)          # e.g. "cuda", "xpu", "cpu"
print(f"Using device : {device}")
print(f"Device name  : {device_name}")


## Step 1 — Import the Dataset

TinyStories is a synthetic dataset of short stories that only contain words a typical 3–4-year-old understands, generated by GPT-3.5 and GPT-4.

In [ ]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")
print(ds)


## Step 2 — Tokenise the Dataset

1. Tokenise every story into token IDs using the GPT-2 tokeniser.
2. Write `train.bin` and `validation.bin` as memory-mapped NumPy arrays on disk so we never load the full dataset into RAM.

In [ ]:
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm

enc = tiktoken.get_encoding("gpt2")

def process(example):
    ids = enc.encode_ordinary(example["text"])
    return {"ids": ids, "len": len(ids)}

if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=["text"],
        desc="tokenizing the splits",
        num_proc=4,
    )
    for split, dset in tokenized.items():
        arr_len = np.sum(dset["len"], dtype=np.uint64)
        filename = f"{split}.bin"
        dtype = np.uint16
        arr = np.memmap(filename, dtype=dtype, mode="w+", shape=(arr_len,))
        total_batches = 1024
        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f"writing {filename}"):
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format("numpy")
            arr_batch = np.concatenate(batch["ids"])
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)
        arr.flush()
    print("Tokenisation complete.")
else:
    print("Token files already exist — skipping.")


## Step 3 — Create Input-Output Batches

In [ ]:
def get_batch(split):
    """Load a random batch from the memory-mapped token files."""
    if split == "train":
        data = np.memmap("train.bin", dtype=np.uint16, mode="r")
    else:
        data = np.memmap("validation.bin", dtype=np.uint16, mode="r")

    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i     : i + block_size    ].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1 : i + 1 + block_size].astype(np.int64)) for i in ix])

    # pin_memory only works for CUDA; fall through to plain .to() for XPU / CPU
    if device_str == "cuda":
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x = x.to(device)
        y = y.to(device)

    return x, y


## Step 4 — Define the SLM Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
import numpy as np
from tqdm.auto import tqdm
from contextlib import nullcontext
import os


class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias   = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn        = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj        = nn.Linear(config.n_embd, config.n_embd,     bias=config.bias)
        self.attn_dropout  = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head        = config.n_head
        self.n_embd        = config.n_embd
        self.flash         = hasattr(F, "scaled_dot_product_attention")
        if not self.flash:
            self.register_buffer(
                "bias",
                torch.tril(torch.ones(config.block_size, config.block_size))
                     .view(1, 1, config.block_size, config.block_size),
            )

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=None,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=True,
            )
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y   = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1  = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2  = LayerNorm(config.n_embd, config.bias)
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


@dataclass
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layer:    int
    n_head:     int
    n_embd:     int
    dropout:    float = 0.0
    bias:       bool  = True


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),
            wpe  = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h    = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device_ = idx.device
        b, t    = idx.size()
        assert t <= self.config.block_size
        pos     = torch.arange(0, t, dtype=torch.long, device=device_)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x       = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss   = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1,
            )
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond  = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits    = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")
            probs    = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx      = torch.cat((idx, idx_next), dim=1)
        return idx


In [ ]:
config = GPTConfig(
    vocab_size = 50257,   # GPT-2 tokeniser vocab size
    block_size = 128,     # context window
    n_layer    = 6,
    n_head     = 6,
    n_embd     = 384,
    dropout    = 0.1,
    bias       = True,
)

model = GPT(config)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}  (~{total_params/1e6:.1f}M)")


## Step 5 — Loss Estimation

In [ ]:
def estimate_loss(model):
    out = {}
    model.eval()
    with torch.inference_mode():
        for split in ["train", "val"]:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                with ctx:
                    _, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
    model.train()
    return out


## Step 6 — Training Configuration

### ⚡ Batch size guide for Intel Arc B570 (~10 GB VRAM)
| `batch_size` | Action |
|---|---|
| **8** | Start here — always safe |
| **16** | Bump if utilisation looks low |
| **32** | Original notebook value; try once 16 is stable |

`gradient_accumulation_steps` compensates so the effective batch stays the same.


In [ ]:
import torch
from contextlib import nullcontext

# ── Hyperparameters ────────────────────────────────────────────────────────────
learning_rate               = 1e-4
max_iters                   = 20000
warmup_steps                = 1000
min_lr                      = 5e-4
eval_iters                  = 500
batch_size                  = 8      # ← start here; increase to 16 → 32
block_size                  = 128
gradient_accumulation_steps = 32

# ── dtype / autocast ──────────────────────────────────────────────────────────
# bfloat16 on CUDA (Ampere+) and XPU; float16 on older CUDA; float32 on CPU
if device_str == "cuda":
    dtype = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
elif device_str == "xpu":
    dtype = "bfloat16"          # Arc supports bfloat16
else:
    dtype = "float32"

ptdtype = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[dtype]

# torch.amp.autocast accepts "cuda", "xpu", or "cpu"
ctx = nullcontext() if device_str == "cpu" else torch.amp.autocast(device_type=device_str, dtype=ptdtype)

torch.manual_seed(42)
print(f"dtype      : {dtype}")
print(f"device_str : {device_str}")
print(f"batch_size : {batch_size}")


## Step 7 — Optimiser & Scheduler

In [ ]:
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.95),
    weight_decay=0.1,
    eps=1e-9,
)

scheduler_warmup = LinearLR(optimizer, total_iters=warmup_steps)
scheduler_decay  = CosineAnnealingLR(optimizer, T_max=max_iters - warmup_steps, eta_min=min_lr)
scheduler        = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps])

# GradScaler: only active for float16 (handles NaN gradients)
# torch.amp.GradScaler works for both "cuda" and "xpu" in PyTorch 2.x
scaler = torch.amp.GradScaler(device=device_str, enabled=(dtype == "float16"))

print("Optimiser ready.")


## Step 8 — Pre-train the SLM

While training runs, verify GPU usage:
- Task Manager → Performance → GPU

You should see compute utilisation rise on the Intel Arc B570.

In [ ]:
best_val_loss           = float("inf")
best_model_params_path  = "best_model_params.pt"
train_loss_list, validation_loss_list = [], []

model = model.to(device)

for epoch in tqdm(range(max_iters)):

    # ── Evaluation ────────────────────────────────────────────────────────────
    if epoch % eval_iters == 0 and epoch != 0:
        losses = estimate_loss(model)
        print(f"Epoch {epoch:>6}: train {losses['train']:.4f} | val {losses['val']:.4f} | "
              f"lr {optimizer.param_groups[0]['lr']:.2e}")
        train_loss_list.append(losses["train"])
        validation_loss_list.append(losses["val"])

        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            torch.save(model.state_dict(), best_model_params_path)
            print(f"  ✔ Saved checkpoint (val loss {best_val_loss:.4f})")

    # ── Forward / backward ────────────────────────────────────────────────────
    X, y = get_batch("train")

    with ctx:
        _, loss = model(X, y)
        loss    = loss / gradient_accumulation_steps
        scaler.scale(loss).backward()

    if ((epoch + 1) % gradient_accumulation_steps == 0) or (epoch + 1 == max_iters):
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    scheduler.step()


## Step 9 — Plot the Loss Curves

In [ ]:
import matplotlib.pyplot as plt

train_loss_cpu = [l.cpu().detach().item() for l in train_loss_list]
val_loss_cpu   = [l.cpu().detach().item() for l in validation_loss_list]

plt.figure(figsize=(9, 4))
plt.plot(train_loss_cpu, "g-", label="Train loss")
plt.plot(val_loss_cpu,   "r-", label="Val loss")
plt.xlabel(f"Eval steps (every {eval_iters} epochs)")
plt.ylabel("Loss")
plt.title("SLM Training — Loss Curves")
plt.legend()
plt.tight_layout()
plt.savefig("loss_curves.png", dpi=150)
plt.show()
print("Saved loss_curves.png")


## Step 10 — Load Checkpoint & Run Inference

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")

# Re-create model and load best weights
model_infer = GPT(config)
model_infer.load_state_dict(
    torch.load(
        "best_model_params.pt",
        map_location=device,   # works for cuda / xpu / cpu
    )
)
model_infer = model_infer.to(device)
model_infer.eval()
print("Checkpoint loaded.")


In [ ]:
def generate(prompt: str, max_new_tokens: int = 200, temperature: float = 0.8, top_k: int = 50):
    context = torch.tensor(enc.encode_ordinary(prompt), dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model_infer.generate(context, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return enc.decode(out.squeeze().tolist())

print(generate("Once upon a time there was a pumpkin."))


In [ ]:
print(generate("A little girl went to the woods"))
